# E-Commerce 360° Analytics — Data Cleaning

## Objective

Clean and standardize the raw Olist datasets while preserving the original raw data.

### Cleaning Steps

1. Load raw datasets
2. Standardize column names
3. Fix data types
4. Handle missing values
5. Handle duplicates
6. Validate categorical values
7. Check invalid values
8. Convert date columns
9. Save processed datasets

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Define project directories

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

# Create processed folder if it does not exist
processed_path.mkdir(parents=True, exist_ok=True)

print("Raw data path:", raw_path.resolve())
print("Processed data path:", processed_path.resolve())

Raw data path: C:\Users\Sam\Documents\PROJECTS\DataAnalyst\E-Commerce-360-Analytics\data\raw
Processed data path: C:\Users\Sam\Documents\PROJECTS\DataAnalyst\E-Commerce-360-Analytics\data\processed


In [3]:
# Find all raw CSV files

csv_files = sorted(raw_path.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

CSV files found: 9


In [4]:
# Load each dataset into a separate DataFrame

dataframes = {}

for file in csv_files:
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    dataframes[name] = pd.read_csv(file)

print(f"Datasets loaded: {len(dataframes)}")

Datasets loaded: 9


In [5]:
for name, df in dataframes.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
product_category_name_translation: (71, 2)


In [7]:
# Create working copies for cleaning

cleaned_dataframes = {
    name: df.copy()
    for name, df in dataframes.items()
}

print(f"Working datasets created: {len(cleaned_dataframes)}")

Working datasets created: 9


In [8]:
# Standardize column names

for name, df in cleaned_dataframes.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

print("Column names standardized successfully.")

Column names standardized successfully.


In [9]:
# Identify columns that contain date or timestamp data

date_columns = {}

for name, df in cleaned_dataframes.items():
    columns = [
        col for col in df.columns
        if "date" in col or "timestamp" in col
    ]

    if columns:
        date_columns[name] = columns

for name, columns in date_columns.items():
    print(f"{name}:")
    for column in columns:
        print(f"  - {column}")

order_items:
  - shipping_limit_date
order_reviews:
  - review_creation_date
  - review_answer_timestamp
orders:
  - order_purchase_timestamp
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date


In [10]:
# Check data types of the identified date columns

for name, columns in date_columns.items():
    print(f"\n{name}")

    for column in columns:
        print(f"{column}: {cleaned_dataframes[name][column].dtype}")


order_items
shipping_limit_date: str

order_reviews
review_creation_date: str
review_answer_timestamp: str

orders
order_purchase_timestamp: str
order_delivered_carrier_date: str
order_delivered_customer_date: str
order_estimated_delivery_date: str


In [11]:
# Convert date and timestamp columns to datetime

for name, columns in date_columns.items():
    for column in columns:
        cleaned_dataframes[name][column] = pd.to_datetime(
            cleaned_dataframes[name][column],
            errors="coerce"
        )

print("Date columns converted successfully.")

Date columns converted successfully.


In [12]:
# Verify converted date data types

for name, columns in date_columns.items():
    print(f"\n{name}")

    for column in columns:
        print(f"{column}: {cleaned_dataframes[name][column].dtype}")


order_items
shipping_limit_date: datetime64[us]

order_reviews
review_creation_date: datetime64[us]
review_answer_timestamp: datetime64[us]

orders
order_purchase_timestamp: datetime64[us]
order_delivered_carrier_date: datetime64[us]
order_delivered_customer_date: datetime64[us]
order_estimated_delivery_date: datetime64[us]


In [13]:
# Create missing-value report

missing_report = []

for name, df in cleaned_dataframes.items():

    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_report.append({
                "Dataset": name,
                "Column": column,
                "Missing_Count": missing_count,
                "Missing_Percentage": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_report = pd.DataFrame(missing_report)

missing_report.sort_values(
    "Missing_Percentage",
    ascending=False
)

,Dataset,Column,Missing_Count,Missing_Percentage
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
4,orders,order_delivered_customer_date,2965,2.98
6,products,product_name_lenght,610,1.85
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [14]:
# Handle product fields with meaningful missing values

products = cleaned_dataframes["products"]

# Missing category
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

# Missing derived product attributes
products["product_name_lenght"] = (
    products["product_name_lenght"]
    .fillna(0)
)

products["product_description_lenght"] = (
    products["product_description_lenght"]
    .fillna(0)
)

products["product_photos_qty"] = (
    products["product_photos_qty"]
    .fillna(0)
)

print("Product missing values handled.")

Product missing values handled.


In [15]:
# Check remaining missing values in products

remaining_missing = products.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print(remaining_missing)

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64


In [16]:
# Fill remaining product measurement missing values with median

measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in measurement_columns:
    products[column] = products[column].fillna(products[column].median())

print("Remaining product measurement values handled.")

Remaining product measurement values handled.


In [17]:
# Verify product missing values

remaining_missing = products.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print(remaining_missing)

Series([], dtype: int64)


In [18]:
# Remove exact duplicate records from cleaned datasets

for name, df in cleaned_dataframes.items():
    before = len(df)

    cleaned_dataframes[name] = df.drop_duplicates().reset_index(drop=True)

    after = len(cleaned_dataframes[name])

    print(f"{name}: {before - after} duplicate rows removed")

customers: 0 duplicate rows removed
geolocation: 261831 duplicate rows removed
order_items: 0 duplicate rows removed
order_payments: 0 duplicate rows removed
order_reviews: 0 duplicate rows removed
orders: 0 duplicate rows removed
products: 0 duplicate rows removed
sellers: 0 duplicate rows removed
product_category_name_translation: 0 duplicate rows removed


In [19]:
# Display dataset sizes after duplicate removal

for name, df in cleaned_dataframes.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (738332, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
product_category_name_translation: (71, 2)


In [20]:
# Validate remaining missing values

for name, df in cleaned_dataframes.items():
    missing_count = df.isna().sum().sum()

    print(f"{name}: {missing_count} missing values")

customers: 0 missing values
geolocation: 0 missing values
order_items: 0 missing values
order_payments: 0 missing values
order_reviews: 145903 missing values
orders: 4908 missing values
products: 0 missing values
sellers: 0 missing values
product_category_name_translation: 0 missing values


In [21]:
# Save cleaned datasets to the processed folder

for name, df in cleaned_dataframes.items():
    output_file = processed_path / f"{name}.csv"
    df.to_csv(output_file, index=False)

    print(f"Saved: {output_file.name}")

Saved: customers.csv
Saved: geolocation.csv
Saved: order_items.csv
Saved: order_payments.csv
Saved: order_reviews.csv
Saved: orders.csv
Saved: products.csv
Saved: sellers.csv
Saved: product_category_name_translation.csv


In [22]:
# final cleaning summary

cleaning_summary = []

for name, df in cleaned_dataframes.items():
    cleaning_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": df.isna().sum().sum(),
        "Duplicate_Rows": df.duplicated().sum()
    })

cleaning_summary = pd.DataFrame(cleaning_summary)

cleaning_summary

,Dataset,Rows,Columns,Missing_Values,Duplicate_Rows
0,customers,99441,5,0,0
1,geolocation,738332,5,0,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,0,0
7,sellers,3095,4,0,0
8,product_category_name_translation,71,2,0,0


In [23]:
# Save cleaning summary

summary_path = processed_path / "data_cleaning_summary.csv"

cleaning_summary.to_csv(
    summary_path,
    index=False
)

print(f"Cleaning summary saved to: {summary_path.resolve()}")

Cleaning summary saved to: C:\Users\Sam\Documents\PROJECTS\DataAnalyst\E-Commerce-360-Analytics\data\processed\data_cleaning_summary.csv


# Data Cleaning Summary

## Cleaning Overview

The raw Olist e-commerce datasets were cleaned and prepared for downstream SQL analysis, Excel reporting, Power BI dashboards, and the AI Business Analytics Assistant.

## Key Cleaning Steps

1. Standardized column names across all datasets.
2. Converted date and timestamp columns to proper datetime format.
3. Handled missing values using business-specific rules.
4. Preserved meaningful missing values in review and delivery-related fields.
5. Filled missing product attributes where appropriate.
6. Removed exact duplicate records.
7. Validated missing values, duplicate records, data types, and empty columns.
8. Saved all cleaned datasets to the `data/processed/` folder.

## Data Quality Decisions

- Review comments were kept as missing when customers did not provide comments.
- Missing delivery dates were preserved because they may indicate undelivered or incomplete orders.
- Missing product categories were labeled as `unknown`.
- Missing product measurement values were handled using median values.
- Exact duplicate records were removed.

## Final Output

The cleaned datasets are stored in:

`data/processed/`

These processed datasets are ready for:

**SQL Database → Business Analysis → Excel/MIS → Power BI → AI Business Analytics Assistant**